# Qwen Fine-Tuning on ROCm -- CFPB Complaint Categorisation

**Task:** Convert unstructured consumer complaint narratives into structured ticket metadata  
`{ product, sub_product, issue, sub_issue }`

**Model:** `Qwen/Qwen2.5-7B-Instruct` fine-tuned with LoRA on AMD Instinct MI300X (192 GB VRAM)  
**Backend:** ROCm 7.2.4 / HIP -- no CUDA, no bitsandbytes  
**Dataset:** Full CFPB Consumer Complaint Database (train / val / test splits)

---

## Notebook Structure

| # | Section |
|---|----------|
| 1 | Environment check |
| 2 | Config & constants |
| 3 | Directory setup |
| 4 | Model & tokenizer loading |
| 5 | Data loading & stratified sampling |
| 6 | Dataset formatting & tokenisation |
| 7 | Evaluation utilities |
| 8 | Baseline qualitative demo |
| 9 | Baseline quantitative evaluation |
| 10 | LoRA setup & training |
| 11 | Post-training qualitative demo |
| 12 | Post-training quantitative evaluation (constrained decoding) |
| 13 | Results comparison |
| 14 | Save adapter |


## 1. Environment & Dependency Check

In [2]:
# ── Install required packages (run once) ─────────────────────────────────────
# Uncomment if running for the first time.

!pip install -q transformers==4.44.0 peft==0.12.0 accelerate==0.34.0 scikit-learn\
             datasets==2.21.0 trl==0.10.1 \
             rouge-score sacrebleu nltk 



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
import torch

# ── ROCm / HIP device check ───────────────────────────────────────────────────
# On ROCm, torch.cuda.* APIs map to AMD GPUs via HIP.
print(f"PyTorch version    : {torch.__version__}")
print(f"ROCm/HIP available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU device         : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will run on CPU (very slow).")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device       : {DEVICE}")


PyTorch version    : 2.10.0+rocm7.2.4.git3d3aa833
ROCm/HIP available : True
GPU device         : AMD Instinct MI300X
GPU memory         : 206.1 GB

Using device       : cuda


## 2. Config & Constants

All tunable parameters live here — nothing else in the notebook needs to change.


In [ ]:
# Paths
BASE_DIR         = "/workspace/shared"
MODEL_SAVE_DIR   = f"{BASE_DIR}/Day3/models/qwen2.5-7b"
ADAPTER_SAVE_DIR = f"{BASE_DIR}/Day3/models/qwen2.5-7b-lora"
CHECKPOINT_DIR   = f"{BASE_DIR}/Day3/checkpoints"
RESULTS_DIR      = f"{BASE_DIR}/Day3/results"

TRAIN_PATH = f"{BASE_DIR}/CFPB-Dataset-for-qwen/train.jsonl"
VAL_PATH   = f"{BASE_DIR}/CFPB-Dataset-for-qwen/validation.jsonl"
TEST_PATH  = f"{BASE_DIR}/CFPB-Dataset-for-qwen/test.jsonl"

# Model
# Upgraded from 1.5B to 7B -- MI300X has 192 GB VRAM so there is no memory constraint.
# The 7B model has more capacity for the CFPB issue taxonomy (80+ canonical label strings).
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Dataset sizes
# N_TRAIN = None means use the full training set (no subsampling).
# Previous runs capped at 3000; the full dataset gives better label coverage.
N_TRAIN = None
N_VAL   = 500
N_TEST  = 500
RANDOM_SEED = 42

# Tokenisation
MAX_SEQ_LENGTH = 1024

# LoRA hyperparameters
# Rank is 16 (down from 32) -- the 7B base model has more capacity per rank
# so a lower rank adapts it efficiently without risk of overfitting.
LORA_R              = 16
LORA_ALPHA          = 32    # keep at 2x rank
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training hyperparameters
# Epochs set to 5 -- previous runs showed val loss plateauing around epoch 2-3.
# Early stopping (patience=3) will halt training automatically if loss stalls.
# Batch size 8 -- MI300X comfortably fits this for a 7B bf16 model.
NUM_EPOCHS           = 5
BATCH_SIZE           = 8
GRADIENT_ACCUM_STEPS = 4     # effective batch size = 8 * 4 = 32
LEARNING_RATE        = 1e-4
LOGGING_STEPS        = 20
EVAL_STEPS           = 100
SAVE_STEPS           = 100

# Inference
MAX_NEW_TOKENS = 128

# Qualitative demo example indices
DEMO_INDICES = [0, 1, 2, 3, 4]


## 3. Directory Setup

In [5]:
import os

def setup_directories(*dirs: str) -> None:
    for d in dirs:
        os.makedirs(d, exist_ok=True)
        print(f"  Ready: {d}")

print("Setting up directories...")
setup_directories(MODEL_SAVE_DIR, ADAPTER_SAVE_DIR, CHECKPOINT_DIR, RESULTS_DIR)


Setting up directories...
  Ready: /workspace/shared/Day3/models/qwen2.5-1.5b
  Ready: /workspace/shared/Day3/models/qwen2.5-1.5b-lora
  Ready: /workspace/shared/Day3/checkpoints
  Ready: /workspace/shared/Day3/results


## 4. Model & Tokenizer Loading

Loaded in **bfloat16** — the native precision for AMD MI-series GPUs on ROCm.  
`bitsandbytes` (4-bit quantisation) is CUDA-only and is intentionally omitted.


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model_and_tokenizer(model_name: str):
    """
    Load Qwen model and tokenizer.

    ROCm note: bfloat16 over float16 — better numerical stability on AMD GPUs.
    bitsandbytes 4-bit quantisation is CUDA-only; omitted here deliberately.
    """
    print(f"Loading tokenizer : {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Qwen models may ship without a pad token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Loading model     : {model_name}")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,   # optimal on ROCm / AMD GPUs
        device_map="auto",
    )

    print(f"Parameters        : {model.num_parameters():,}")
    return model, tokenizer


model, tokenizer = load_model_and_tokenizer(MODEL_NAME)


Loading tokenizer : Qwen/Qwen2.5-1.5B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading model     : Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Parameters        : 1,543,714,304


## 5. Data Loading & Sampling

Each JSONL line contains a `"messages"` key — a list of chat turns  
`[{"role": "system"|"user"|"assistant", "content": "..."}]`.

The **assistant turn** is the structured JSON output the model must learn to produce.


In [ ]:
import json
import random
from collections import defaultdict
from typing import List, Dict, Any


def load_jsonl(path: str) -> List[Dict[str, Any]]:
    """Read a JSONL file and return a list of parsed JSON objects."""
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data


def get_label_key(example: Dict) -> str:
    """
    Extract a stratification key (product + issue) from one example.

    Combining product and issue catches long-tail combinations that
    would be missed by stratifying on product alone.
    """
    try:
        ref     = json.loads(example["messages"][-1]["content"])
        product = ref.get("product", "unknown").strip().lower()
        issue   = ref.get("issue",   "unknown").strip().lower()
        return f"{product}|{issue}"
    except Exception:
        return "unknown|unknown"


def stratified_sample(data: List[Dict], n, seed: int = RANDOM_SEED) -> List[Dict]:
    """
    Sample n examples with proportional stratification by product+issue.
    If n is None, return the full dataset shuffled.

    Steps:
      1. Group examples into buckets by (product, issue).
      2. Allocate samples proportionally -- every bucket gets at least 1.
      3. Top up to exactly n from the leftover pool if needed.
    """
    random.seed(seed)

    if n is None:
        shuffled = data[:]
        random.shuffle(shuffled)
        return shuffled

    buckets = defaultdict(list)
    for ex in data:
        buckets[get_label_key(ex)].append(ex)

    print(f"  Unique product+issue combinations: {len(buckets)}")

    result, leftover = [], []
    for bucket in buckets.values():
        alloc  = max(1, round(n * len(bucket) / len(data)))
        alloc  = min(alloc, len(bucket))
        chosen = random.sample(bucket, alloc)
        result.extend(chosen)
        chosen_ids = {id(x) for x in chosen}
        leftover.extend(x for x in bucket if id(x) not in chosen_ids)

    if len(result) < n and leftover:
        extra = random.sample(leftover, min(n - len(result), len(leftover)))
        result.extend(extra)

    random.shuffle(result)
    return result[:n]


# Load raw data
print("Loading dataset splits...")
raw_train = load_jsonl(TRAIN_PATH)
raw_val   = load_jsonl(VAL_PATH)
raw_test  = load_jsonl(TEST_PATH)

print(f"  Full train : {len(raw_train):,}")
print(f"  Full val   : {len(raw_val):,}")
print(f"  Full test  : {len(raw_test):,}")

# Train: stratified over full dataset; val/test: random sample
print("\nSampling splits...")
train_data = stratified_sample(raw_train, N_TRAIN)

random.seed(RANDOM_SEED)
val_data  = random.sample(raw_val,  min(N_VAL,  len(raw_val)))
test_data = random.sample(raw_test, min(N_TEST, len(raw_test)))

print(f"\nFinal sizes -- train: {len(train_data):,} | val: {len(val_data)} | test: {len(test_data)}")

print("\nSample training example:")
for turn in train_data[0]["messages"]:
    print(f"  [{turn['role'].upper()}] {turn['content'][:120]}")


## 6. Dataset Formatting & Tokenisation

We apply the Qwen **chat template** (which inserts `<|im_start|>` / `<|im_end|>` tokens)  
then tokenise. All non-tensor columns (`messages`, `text`) are dropped so the  
DataCollator only sees `input_ids` and `attention_mask`.


In [8]:
from datasets import Dataset

def format_chat(example: Dict[str, Any]) -> Dict[str, str]:
    """Apply Qwen chat template — converts message list to a single training string."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,   # assistant turn already present
    )
    return {"text": text}


def tokenize_example(example: Dict[str, Any]) -> Dict:
    """
    Tokenise the formatted string.
    padding=False — padding happens per-batch in the DataCollator (more efficient).
    """
    return tokenizer(
        example["text"],
        truncation=True,
        padding=False,
        max_length=MAX_SEQ_LENGTH,
    )


def build_hf_dataset(data: List[Dict]) -> Dataset:
    """
    Build a HuggingFace Dataset ready for the Trainer.

    The column-drop step is critical: the raw 'messages' column (list of dicts)
    and 'text' column (string) cause a ValueError when the DataCollator tries
    to batch them into tensors. Keeping only tokenized columns fixes this.
    """
    ds = Dataset.from_list(data)
    ds = ds.map(format_chat)
    ds = ds.map(tokenize_example, batched=True)

    keep = {"input_ids", "attention_mask", "labels"}
    ds   = ds.remove_columns([c for c in ds.column_names if c not in keep])
    return ds


print("Formatting and tokenising datasets...")
train_ds = build_hf_dataset(train_data)
val_ds   = build_hf_dataset(val_data)

print(f"  Train : {len(train_ds)} examples | columns: {train_ds.column_names}")
print(f"  Val   : {len(val_ds)} examples")
print(f"\nFirst example — first 20 token ids: {train_ds[0]['input_ids'][:20]}")


Formatting and tokenising datasets...


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

  Train : 3000 examples | columns: ['input_ids', 'attention_mask']
  Val   : 250 examples

First example — first 20 token ids: [151644, 8948, 271, 2610, 525, 264, 22798, 12181, 22881, 17847, 382, 2082, 55856, 279, 12181, 19221, 323, 10542, 1447, 16]


## 7. Evaluation Utilities

### Two complementary evaluation layers

| Layer | Metrics | Why it matters |
|-------|---------|----------------|
| **Structured / Field-level** | Exact Match per field, Field Accuracy, Exact JSON Match, Micro/Macro/Weighted F1 | The model's job is to produce correct ticket metadata — field accuracy directly measures business value |
| **Generative** | ROUGE-1/2/L, BLEU, SacreBLEU, METEOR | Measures output fluency and n-gram overlap; useful for regression and catching degradation |

> **Why both?** A prediction of `"Checking account"` vs `"Checking or savings account"` scores ~0 on BLEU  
> but may be partially correct at the field level. Using only generative metrics on a structured extraction  
> task gives a misleading picture.


In [ ]:
import re
import json as _json
import numpy as np
from rouge_score import rouge_scorer as _rouge
import sacrebleu as _sacrebleu
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

JSON_FIELDS = ["product", "sub_product", "issue", "sub_issue"]


# Inference helpers

def build_inference_prompt(messages: List[Dict]) -> str:
    """Strip the final assistant turn; model must generate it."""
    return tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True,
    )


def predict(messages: List[Dict], model) -> str:
    """Greedy decode -- returns only the newly generated tokens."""
    prompt = build_inference_prompt(messages)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    model.eval()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True)


def extract_reference(messages: List[Dict]) -> str:
    """Ground-truth: content of the last (assistant) message."""
    return messages[-1]["content"]


# JSON parsing

def parse_json_output(text: str) -> Dict[str, str]:
    """
    Parse a JSON dict from model output.
    Handles markdown fences and minor noise. Returns empty dict on failure.
    """
    text = re.sub(r"```(?:json)?", "", text).strip().rstrip("`").strip()

    try:
        parsed = _json.loads(text)
        if isinstance(parsed, dict):
            return {k: str(v).strip().lower() for k, v in parsed.items()}
    except _json.JSONDecodeError:
        pass

    match = re.search(r"\{[^{}]+\}", text, re.DOTALL)
    if match:
        try:
            parsed = _json.loads(match.group())
            if isinstance(parsed, dict):
                return {k: str(v).strip().lower() for k, v in parsed.items()}
        except _json.JSONDecodeError:
            pass

    return {}


# Constrained decoding

def build_valid_label_vocab(data: List[Dict]) -> Dict[str, set]:
    """
    Collect every canonical CFPB label seen in the dataset for each field.

    The CFPB taxonomy has 80+ issue labels with similar phrasing.
    Building this vocab lets us snap model outputs to valid labels,
    correcting minor lexical differences without changing meaning.
    """
    vocab = {f: set() for f in JSON_FIELDS}
    for example in data:
        try:
            ref = _json.loads(example["messages"][-1]["content"])
            for field in vocab:
                val = ref.get(field, "")
                if val:
                    vocab[field].add(str(val).strip().lower())
        except Exception:
            pass
    for field, vals in vocab.items():
        print(f"  {field:<15}: {len(vals)} unique labels")
    return vocab


def build_tfidf_snappers(vocab: Dict[str, set]) -> Dict:
    """
    Pre-compute TF-IDF vectors for each field label vocabulary.

    TF-IDF cosine similarity distinguishes near-duplicate labels much better
    than Jaccard. For example, "problem with fees" vs "other fee" share most
    words -- Jaccard scores them similarly, TF-IDF on bigrams separates them.

    Returns a dict: {field: (label_list, vectorizer, tfidf_matrix)}
    """
    snappers = {}
    for field, labels in vocab.items():
        label_list = sorted(labels)
        if not label_list:
            continue
        vec    = TfidfVectorizer(ngram_range=(1, 2)).fit(label_list)
        matrix = vec.transform(label_list)
        snappers[field] = (label_list, vec, matrix)
    return snappers


def snap_to_vocab(predicted_val: str, field: str, snappers: Dict) -> str:
    """
    Return the canonical label closest to predicted_val via cosine similarity.
    If predicted_val is already valid, return it unchanged.
    """
    if field not in snappers:
        return predicted_val

    label_list, vectorizer, matrix = snappers[field]

    if predicted_val in label_list:
        return predicted_val

    query = vectorizer.transform([predicted_val])
    sims  = cosine_similarity(query, matrix)[0]
    return label_list[int(np.argmax(sims))]


def constrained_predict(messages: List[Dict], model, vocab: Dict, snappers: Dict) -> str:
    """
    Two-pass prediction:
      Pass 1 -- model generates a JSON string via greedy decode.
      Pass 2 -- each output field is snapped to the nearest canonical
                CFPB label using TF-IDF cosine similarity.

    Falls back to raw model output if JSON parsing fails entirely.
    """
    raw_output = predict(messages, model)

    try:
        clean = re.sub(r"```(?:json)?", "", raw_output).strip().strip("`").strip()
        match = re.search(r"\{[^{}]+\}", clean, re.DOTALL)
        obj   = _json.loads(match.group() if match else clean)
    except Exception:
        return raw_output

    snapped = {}
    for field in JSON_FIELDS:
        predicted_val  = str(obj.get(field, "not specified")).strip().lower()
        snapped[field] = snap_to_vocab(predicted_val, field, snappers)

    return _json.dumps(snapped)


# Metric computation

def compute_structured_metrics(predictions: List[str], references: List[str]) -> Dict:
    """
    Field-level metrics for structured JSON extraction.

    Returns exact JSON match, per-field accuracy, micro/macro/weighted F1.
    These are the primary metrics for this task.
    """
    parsed_preds = [parse_json_output(p) for p in predictions]
    parsed_refs  = [parse_json_output(r) for r in references]

    exact_json = sum(p == r for p, r in zip(parsed_preds, parsed_refs)) / len(predictions)

    field_acc = {}
    for field in JSON_FIELDS:
        correct = sum(
            p.get(field, "__missing__") == r.get(field, "__missing__")
            for p, r in zip(parsed_preds, parsed_refs)
        )
        field_acc[field] = round(correct / len(predictions), 4)

    avg_field_acc = round(sum(field_acc.values()) / len(JSON_FIELDS), 4)

    flat_preds, flat_refs = [], []
    for p_dict, r_dict in zip(parsed_preds, parsed_refs):
        for field in JSON_FIELDS:
            flat_preds.append(p_dict.get(field, "__missing__"))
            flat_refs.append(r_dict.get(field, "__missing__"))

    return {
        "exact_json_match"  : round(exact_json, 4),
        "avg_field_accuracy": avg_field_acc,
        "field_accuracy"    : field_acc,
        "micro_f1"   : round(f1_score(flat_refs, flat_preds, average="micro",    zero_division=0), 4),
        "macro_f1"   : round(f1_score(flat_refs, flat_preds, average="macro",    zero_division=0), 4),
        "weighted_f1": round(f1_score(flat_refs, flat_preds, average="weighted", zero_division=0), 4),
    }


def compute_generative_metrics(predictions: List[str], references: List[str]) -> Dict:
    """
    ROUGE, BLEU, SacreBLEU, METEOR -- secondary metrics.
    Useful for tracking regression and output fluency.
    """
    import nltk
    nltk.download("wordnet", quiet=True)
    from nltk.translate.meteor_score import meteor_score

    scorer = _rouge.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    r1, r2, rL, met = [], [], [], []

    for pred, ref in zip(predictions, references):
        s = scorer.score(ref, pred)
        r1.append(s["rouge1"].fmeasure)
        r2.append(s["rouge2"].fmeasure)
        rL.append(s["rougeL"].fmeasure)
        met.append(meteor_score([ref.split()], pred.split()))

    bleu  = corpus_bleu(
        [[r.split()] for r in references],
        [p.split() for p in predictions],
        smoothing_function=SmoothingFunction().method1,
    )
    sacre = _sacrebleu.corpus_bleu(predictions, [references])
    avg   = lambda lst: round(sum(lst) / len(lst), 4)

    return {
        "rouge1"   : avg(r1),
        "rouge2"   : avg(r2),
        "rougeL"   : avg(rL),
        "bleu"     : round(bleu, 4),
        "sacrebleu": round(sacre.score, 2),
        "meteor"   : avg(met),
    }


def evaluate_model(model, data: List[Dict], label: str = "Evaluation") -> Dict:
    """Run plain greedy inference + metrics. Used for baseline evaluation."""
    predictions, references = [], []
    for example in tqdm(data, desc=label):
        predictions.append(predict(example["messages"], model))
        references.append(extract_reference(example["messages"]))
    return {
        "structured"   : compute_structured_metrics(predictions, references),
        "generative"   : compute_generative_metrics(predictions, references),
        "_predictions" : predictions,
        "_references"  : references,
    }


def save_metrics(metrics: Dict, path: str) -> None:
    """Write metrics to JSON, skipping internal _ keys."""
    clean = {k: v for k, v in metrics.items() if not k.startswith("_")}
    with open(path, "w") as f:
        _json.dump(clean, f, indent=2)
    print(f"Saved: {path}")


## 8. Qualitative Inference Demo — Baseline Model

Before running numbers, see exactly what the **untrained** model produces  
for a handful of real complaints. This makes the quantitative improvement  
tangible and interpretable.


In [10]:
def run_qualitative_demo(model, data: List[Dict], indices: List[int], label: str) -> None:
    """
    Print a side-by-side view of complaint → prediction vs reference
    for a small set of hand-picked examples.
    """
    print(f"\n{'═'*70}")
    print(f"  QUALITATIVE DEMO — {label}")
    print(f"{'═'*70}")

    for i, idx in enumerate(indices):
        example  = data[idx]
        messages = example["messages"]

        # Extract the user complaint (typically the last user turn)
        user_content = next(
            (m["content"] for m in reversed(messages) if m["role"] == "user"), ""
        )
        reference = extract_reference(messages)
        prediction = predict(messages, model)

        # Parse both to check field alignment
        pred_parsed = parse_json_output(prediction)
        ref_parsed  = parse_json_output(reference)

        print(f"\n{'─'*70}")
        print(f"  Example {i+1} (index {idx})")
        print(f"{'─'*70}")

        # Show a trimmed version of the complaint
        complaint_preview = user_content[:400].replace("\n", " ").strip()
        if len(user_content) > 400:
            complaint_preview += "..."
        print(f"\n  COMPLAINT:\n  {complaint_preview}")

        print(f"\n  REFERENCE (ground truth):")
        print(f"  {reference.strip()}")

        print(f"\n  MODEL PREDICTION:")
        print(f"  {prediction.strip()}")

        # Field-level match summary
        if ref_parsed:
            print(f"\n  FIELD MATCH:")
            for field in JSON_FIELDS:
                ref_val  = ref_parsed.get(field, "—")
                pred_val = pred_parsed.get(field, "MISSING")
                match    = "✓" if ref_val == pred_val else "✗"
                print(f"    {match} {field:<14} ref='{ref_val}'   pred='{pred_val}'")

    print(f"\n{'═'*70}\n")


# Run the baseline demo before any training
demo_indices = DEMO_INDICES if DEMO_INDICES else list(range(5))
run_qualitative_demo(model, test_data, demo_indices, label="BASELINE (pre-training)")



══════════════════════════════════════════════════════════════════════
  QUALITATIVE DEMO — BASELINE (pre-training)
══════════════════════════════════════════════════════════════════════


/opt/venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



──────────────────────────────────────────────────────────────────────
  Example 1 (index 0)
──────────────────────────────────────────────────────────────────────

  COMPLAINT:
  Analyze the following customer complaint and determine the appropriate complaint classification.  Customer Complaint: XXXX lowered my credit limit by XXXX with no notice. Account has been open since 2012, and balance had been paid in full for most billing cycles. In this instance, payment was 6 days late. Company claims they notified me of this policy in 2012.

  REFERENCE (ground truth):
  {"product": "Credit card", "sub_product": "Not specified", "issue": "Credit line increase/decrease", "sub_issue": "Not specified"}

  MODEL PREDICTION:
  ```json
{
  "Product": "Credit Card",
  "Sub-product": "Standard Credit Card",
  "Issue": "Credit Limit Reduction Without Notice",
  "Sub-issue": "No Notification Before Action"
}
```

  FIELD MATCH:
    ✗ product        ref='credit card'   pred='MISSING'
    ✗ sub_produ

/opt/venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



──────────────────────────────────────────────────────────────────────
  Example 2 (index 1)
──────────────────────────────────────────────────────────────────────

  COMPLAINT:
  Analyze the following customer complaint and determine the appropriate complaint classification.  Customer Complaint: Freedom Mortgage loan # XXXX This account has always been on auto draft and should not be reporting any late payments. My payments have been drafted at the end of the previous month. The payment for XX/XX/2022 was drafted at the end of XX/XX/2022. My payments were received however...

  REFERENCE (ground truth):
  {"product": "Mortgage", "sub_product": "Conventional home mortgage", "issue": "Incorrect information on your report", "sub_issue": "Account status incorrect"}

  MODEL PREDICTION:
  ```json
{
  "Product": "Mortgage Loan",
  "Sub-product": "Auto Draft Account",
  "Issue": "Incorrect Payment Application",
  "Sub-issue": "Late Payments"
}
```

  FIELD MATCH:
    ✗ product        ref='m

## 9. Baseline Quantitative Evaluation

Full test-set evaluation on the untrained base model.  
These scores are the benchmark — every metric must improve after fine-tuning.


In [11]:
print("Running baseline evaluation on full test set...")
baseline_results = evaluate_model(model, test_data, label="Baseline")

print("\n── Structured Metrics (baseline) ──────────────────────────────────────")
s = baseline_results["structured"]
print(f"  Exact JSON Match    : {s['exact_json_match']:.4f}")
print(f"  Avg Field Accuracy  : {s['avg_field_accuracy']:.4f}")
for field, acc in s["field_accuracy"].items():
    print(f"    {field:<16}: {acc:.4f}")
print(f"  Micro F1            : {s['micro_f1']:.4f}")
print(f"  Macro F1            : {s['macro_f1']:.4f}")
print(f"  Weighted F1         : {s['weighted_f1']:.4f}")

print("\n── Generative Metrics (baseline) ──────────────────────────────────────")
g = baseline_results["generative"]
for k, v in g.items():
    print(f"  {k:<12}: {v}")

save_metrics(baseline_results, f"{RESULTS_DIR}/baseline_metrics.json")


Running baseline evaluation on full test set...


Baseline: 100%|██████████| 250/250 [04:30<00:00,  1.08s/it]



── Structured Metrics (baseline) ──────────────────────────────────────
  Exact JSON Match    : 0.0000
  Avg Field Accuracy  : 0.0000
    product         : 0.0000
    sub_product     : 0.0000
    issue           : 0.0000
    sub_issue       : 0.0000
  Micro F1            : 0.0000
  Macro F1            : 0.0000
  Weighted F1         : 0.0000

── Generative Metrics (baseline) ──────────────────────────────────────
  rouge1      : 0.4333
  rouge2      : 0.2239
  rougeL      : 0.3998
  bleu        : 0.0003
  sacrebleu   : 15.13
  meteor      : 0.1027
Saved: /workspace/shared/Day3/results/baseline_metrics.json


## 10. LoRA Setup & Training

**LoRA (Low-Rank Adaptation)** freezes the base model and injects small trainable matrices (rank `r`) into the attention layers. Only ~0.5% of parameters are updated, keeping training fast and VRAM usage low.

### Hardware and precision choices (MI300X)
- `bf16=True` -- native AMD CDNA3 precision, numerically stable and fast  
- `fp16=False` -- disabled; can be unstable on some AMD setups  
- `optim=adamw_torch` -- replaces `paged_adamw_8bit` (bitsandbytes is CUDA-only)  
- `batch_size=8` -- MI300X has 192 GB VRAM; no memory pressure at 7B in bf16  
- **Early stopping with `patience=3`** -- halts training if val loss does not improve  
  for 3 consecutive eval checkpoints, preventing wasted compute


In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import (
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
)


def setup_lora(model, r, alpha, dropout, targets):
    """
    Attach LoRA adapters to the specified attention projection layers.
    Base model weights are frozen -- only the A/B adapter matrices train.
    """
    config = LoraConfig(
        r              = r,
        lora_alpha     = alpha,
        lora_dropout   = dropout,
        bias           = "none",
        task_type      = "CAUSAL_LM",
        target_modules = targets,
    )
    peft_model = get_peft_model(model, config)
    peft_model.print_trainable_parameters()
    return peft_model


def build_training_args(output_dir: str) -> TrainingArguments:
    """
    TrainingArguments configured for ROCm / AMD MI300X.

    Key choices:
      bf16=True             -- native MI300X precision
      optim=adamw_torch     -- no bitsandbytes dependency
      load_best_model_at_end -- restores the checkpoint with lowest val loss
    """
    return TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = NUM_EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRADIENT_ACCUM_STEPS,
        learning_rate               = LEARNING_RATE,
        bf16                        = True,
        fp16                        = False,
        optim                       = "adamw_torch",
        logging_steps               = LOGGING_STEPS,
        eval_steps                  = EVAL_STEPS,
        save_steps                  = SAVE_STEPS,
        eval_strategy               = "steps",
        save_strategy               = "steps",
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        report_to                   = "none",
        dataloader_num_workers      = 0,
        remove_unused_columns       = False,
    )


# Build label vocab and TF-IDF snappers before training
# Collecting from training data ensures the constrained decoder
# only uses labels the model actually trained on.
print("Building canonical CFPB label vocabulary...")
LABEL_VOCAB = build_valid_label_vocab(train_data)

print("\nBuilding TF-IDF snappers for constrained decoding...")
SNAPPERS = build_tfidf_snappers(LABEL_VOCAB)
print("  Done.")

# Attach LoRA adapters
model = setup_lora(
    model,
    r       = LORA_R,
    alpha   = LORA_ALPHA,
    dropout = LORA_DROPOUT,
    targets = LORA_TARGET_MODULES,
)

# DataCollatorForLanguageModeling pads each batch dynamically and sets
# labels = input_ids (standard causal LM objective). mlm=False disables
# masked LM masking -- we are doing causal LM on a decoder-only model.
data_collator = DataCollatorForLanguageModeling(
    tokenizer          = tokenizer,
    mlm                = False,
    pad_to_multiple_of = 8,
)

training_args = build_training_args(CHECKPOINT_DIR)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    data_collator = data_collator,
    callbacks     = [EarlyStoppingCallback(early_stopping_patience=3)],
)

print("\nStarting training...")
trainer.train()
print("Training complete.")


## 11. Qualitative Inference Demo — Fine-Tuned Model

Same examples, same format. Compare directly against Section 8 to see  
whether field predictions improved, and by how much.


In [18]:
# Use the exact same indices so the comparison is apples-to-apples
run_qualitative_demo(model, test_data, demo_indices, label="FINE-TUNED (post-training)")



══════════════════════════════════════════════════════════════════════
  QUALITATIVE DEMO — FINE-TUNED (post-training)
══════════════════════════════════════════════════════════════════════


/opt/venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



──────────────────────────────────────────────────────────────────────
  Example 1 (index 0)
──────────────────────────────────────────────────────────────────────

  COMPLAINT:
  Analyze the following customer complaint and determine the appropriate complaint classification.  Customer Complaint: XXXX lowered my credit limit by XXXX with no notice. Account has been open since 2012, and balance had been paid in full for most billing cycles. In this instance, payment was 6 days late. Company claims they notified me of this policy in 2012.

  REFERENCE (ground truth):
  {"product": "Credit card", "sub_product": "Not specified", "issue": "Credit line increase/decrease", "sub_issue": "Not specified"}

  MODEL PREDICTION:
  {"product": "Credit card", "sub_product": "Not specified", "issue": "Other fee", "sub_issue": "Not specified"}

  FIELD MATCH:
    ✓ product        ref='credit card'   pred='credit card'
    ✓ sub_product    ref='not specified'   pred='not specified'
    ✗ issue         

## 12. Post-Training Quantitative Evaluation

In [ ]:
# Post-Training Quantitative Evaluation -- Constrained Decoding
#
# We use constrained_predict here instead of plain predict.
# The two-pass approach:
#   1. Model generates a JSON string via greedy decode.
#   2. Each field value is snapped to the nearest canonical CFPB label
#      using TF-IDF cosine similarity (better than Jaccard for near-
#      duplicate labels like "problem with fees" vs "other fee").
#
# Baseline used plain predict -- keeping them separate lets the comparison
# table show the combined effect of fine-tuning + constrained decoding.

print("Running post-training evaluation (constrained decoding)...")

final_predictions, final_references = [], []
for example in tqdm(test_data, desc="Fine-tuned"):
    pred = constrained_predict(example["messages"], model, LABEL_VOCAB, SNAPPERS)
    ref  = extract_reference(example["messages"])
    final_predictions.append(pred)
    final_references.append(ref)

# Package into the same structure evaluate_model returns
# so print_comparison and save_metrics work without modification.
final_results = {
    "structured"   : compute_structured_metrics(final_predictions, final_references),
    "generative"   : compute_generative_metrics(final_predictions, final_references),
    "_predictions" : final_predictions,
    "_references"  : final_references,
}

print("\n-- Structured Metrics (fine-tuned) ---------------------------------")
s = final_results["structured"]
print(f"  Exact JSON Match    : {s['exact_json_match']:.4f}")
print(f"  Avg Field Accuracy  : {s['avg_field_accuracy']:.4f}")
for field, acc in s["field_accuracy"].items():
    print(f"    {field:<16}: {acc:.4f}")
print(f"  Micro F1            : {s['micro_f1']:.4f}")
print(f"  Macro F1            : {s['macro_f1']:.4f}")
print(f"  Weighted F1         : {s['weighted_f1']:.4f}")

print("\n-- Generative Metrics (fine-tuned) ----------------------------------")
for k, v in final_results["generative"].items():
    print(f"  {k:<12}: {v}")

save_metrics(final_results, f"{RESULTS_DIR}/final_metrics.json")


## 13. Results Comparison — Baseline vs Fine-Tuned

In [20]:
def print_comparison(baseline: Dict, final: Dict) -> None:
    """Print a formatted side-by-side metric comparison with delta."""

    def delta_str(b, f):
        d    = f - b
        sign = "+" if d >= 0 else ""
        return f"{sign}{d:.4f}"

    print("\n" + "═"*70)
    print("  STRUCTURED METRICS")
    print("═"*70)
    print(f"  {'Metric':<24}  {'Baseline':>10}  {'Fine-tuned':>12}  {'Δ Delta':>10}")
    print("  " + "─"*60)

    bs = baseline["structured"]
    fs = final["structured"]

    flat_s = {
        "Exact JSON Match"    : (bs["exact_json_match"],  fs["exact_json_match"]),
        "Avg Field Accuracy"  : (bs["avg_field_accuracy"], fs["avg_field_accuracy"]),
        "Micro F1"            : (bs["micro_f1"],           fs["micro_f1"]),
        "Macro F1"            : (bs["macro_f1"],           fs["macro_f1"]),
        "Weighted F1"         : (bs["weighted_f1"],        fs["weighted_f1"]),
    }
    for k, (b, f) in flat_s.items():
        print(f"  {k:<24}  {b:>10.4f}  {f:>12.4f}  {delta_str(b,f):>10}")

    print("\n  Field Accuracy (per field):")
    for field in JSON_FIELDS:
        b = bs["field_accuracy"][field]
        f = fs["field_accuracy"][field]
        print(f"    {field:<20}  {b:>10.4f}  {f:>12.4f}  {delta_str(b,f):>10}")

    print("\n" + "═"*70)
    print("  GENERATIVE METRICS")
    print("═"*70)
    print(f"  {'Metric':<24}  {'Baseline':>10}  {'Fine-tuned':>12}  {'Δ Delta':>10}")
    print("  " + "─"*60)

    bg = baseline["generative"]
    fg = final["generative"]

    for k in bg:
        b, f = bg[k], fg[k]
        print(f"  {k:<24}  {b:>10.4f}  {f:>12.4f}  {delta_str(b,f):>10}")

    print("═"*70 + "\n")


print_comparison(baseline_results, final_results)

# Save combined comparison
save_metrics(
    {"baseline": baseline_results, "fine_tuned": final_results},
    f"{RESULTS_DIR}/metrics_comparison.json",
)



══════════════════════════════════════════════════════════════════════
  STRUCTURED METRICS
══════════════════════════════════════════════════════════════════════
  Metric                      Baseline    Fine-tuned     Δ Delta
  ────────────────────────────────────────────────────────────
  Exact JSON Match              0.0000        0.1600     +0.1600
  Avg Field Accuracy            0.0000        0.5270     +0.5270
  Micro F1                      0.0000        0.5270     +0.5270
  Macro F1                      0.0000        0.1325     +0.1325
  Weighted F1                   0.0000        0.4895     +0.4895

  Field Accuracy (per field):
    product                   0.0000        0.8880     +0.8880
    sub_product               0.0000        0.5440     +0.5440
    issue                     0.0000        0.2000     +0.2000
    sub_issue                 0.0000        0.4760     +0.4760

══════════════════════════════════════════════════════════════════════
  GENERATIVE METRICS
═══════

## 14. Save LoRA Adapter

Only the adapter weights are saved (~10-50 MB vs several GB for the full model).  
To reload for inference, use `PeftModel.from_pretrained(base_model, adapter_path)`.


In [21]:
def save_adapter(model, tokenizer, save_path: str) -> None:
    """Save the LoRA adapter weights and tokenizer."""
    print(f"Saving adapter to {save_path} ...")
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print("Saved.")


save_adapter(model, tokenizer, ADAPTER_SAVE_DIR)


Saving adapter to /workspace/shared/Day3/models/qwen2.5-1.5b-lora ...
Saved.


In [22]:
# ── Reload snippet (for future inference) ─────────────────────────────────────
# from peft import PeftModel
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch
#
# base  = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto")
# tok   = AutoTokenizer.from_pretrained(ADAPTER_SAVE_DIR)
# model = PeftModel.from_pretrained(base, ADAPTER_SAVE_DIR)
# model.eval()
#
# Then call: predict(messages, model)

print("All done.")
print(f"  Adapter  → {ADAPTER_SAVE_DIR}")
print(f"  Results  → {RESULTS_DIR}")


All done.
  Adapter  → /workspace/shared/Day3/models/qwen2.5-1.5b-lora
  Results  → /workspace/shared/Day3/results
